<div align="center">

# Encode a file and pick up where you stopped

Turn a text file into vectors and continue after an interruption.

</div>

## When do you need this?

You have an export of reviews, articles, or support messages.
The file may be too large for one in-memory matrix, or the job may stop halfway through.
**Save completed work** so a restart can continue from the first missing row.

## The story in pictures

![A text file feeds small batches into fse and a local vector store. A restart skips rows already saved.](media/03-files-with-resume.svg)

## 1. Prepare a real text file

We use 1,000 articles from the [AG News test split](https://huggingface.co/datasets/SetFit/ag_news) and POTION-base-8M.
**One line owns one row**, including an empty line.
The [shared setup](README.md#run-a-recipe) installs the pinned packages and explains the temporary workspace.

In [1]:
from _helpers import Recipe
import fse
import numpy as np

recipe = Recipe()
work = recipe.work

Recipe run: 2026-11-01. Generated files are temporary.
Apple M1 Pro. macOS-27.0.1-arm64-arm-64bit-Mach-O. Python 3.13.16.


Load POTION-base-8M and read a few real articles.
Print an article and the vector size to check the connection.

In [2]:
import fse

encoder = recipe.encoder()
texts = recipe.news(12)
print(texts[0])
print("Vector dimensions:", encoder.output_dim)

/Users/oliverborchers/.herdr/worktrees/Fast_Sentence_Embeddings/dr195-tutorials/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.
Vector dimensions: 256


Write the real articles into a UTF-8 file, with one article per line.
Keep the vector store in a separate folder so its saved row count can guide a restart.

In [3]:
texts = recipe.news(1000)
source = work / "articles.txt"
source.write_text("\n".join(t.replace("\n", " ") for t in texts) + "\n", encoding="utf-8")
store_path = work / "article-vectors"
print("Input articles:", len(texts))

Input articles: 1000


## 2. Read small batches

This reader holds at most 200 lines at a time and rejects records larger than 64 KiB.
The `skip` argument tells it how many saved rows to pass over.
**Keep the input unchanged** between runs, so row numbers keep the same meaning.

In [4]:
MAX_RECORD_BYTES = 64 * 1024


def line_batches(path, skip, size=200):
    batch = []
    with path.open("rb") as stream:
        index = 0
        while raw := stream.readline(MAX_RECORD_BYTES + 1):
            if len(raw) > MAX_RECORD_BYTES:
                raise ValueError(f"line {index} exceeds {MAX_RECORD_BYTES} bytes")
            text = raw.rstrip(b"\n").decode("utf-8")
            if index >= skip:
                batch.append(text)
                if len(batch) == size:
                    yield batch
                    batch = []
            index += 1
    if batch:
        yield batch

## 3. Continue from the saved row count

`ShardSink` writes groups of rows to disk, called **shards**.
Its `first_row` tells us how many rows the store already has.
A **durable** row has been saved so it survives a process crash.
During a run, `durable_rows` counts only newly durable rows, so add it to `first_row`.

In [5]:
def resume(stop_after=None):
    with fse.ShardSink(store_path, encoder, shard_rows=200) as sink:
        start = sink.first_row  # existing store prefix, not this run's durable count
        stream = encoder.encode_batches(line_batches(source, start), sink=sink, workers=1)
        try:
            for number, batch in enumerate(stream, 1):
                assert batch.durable_rows is not None
                # Only this prefix can be acknowledged before close.
                confirmed = start + batch.durable_rows
                print("Durable rows:", confirmed)
                if stop_after is not None and number == stop_after:
                    break
        finally:
            stream.close()
    with fse.VectorStore.open(store_path) as store:
        return len(store)

## 4. Stop, restart, and repeat

The first call stops after two batches.
The second finishes the file, and the third proves that **rerunning adds no duplicate rows**.
The final check compares every stored vector with fresh encoder output.

In [6]:
first = resume(stop_after=2)
second = resume()
third = resume()
assert first == 400 and second == third == 1000
with fse.VectorStore.open(store_path) as store:
    offset = 0
    for batch in line_batches(source, 0):
        actual = store.get_vectors(start=offset, stop=offset + len(batch))
        np.testing.assert_array_equal(actual, encoder.encode(batch, workers=1))
        offset += len(batch)
    assert offset == len(store)
print({"interrupted_rows": first, "resumed_rows": second, "rerun_rows": third})

Durable rows: 200
Durable rows: 400
Durable rows: 600
Durable rows: 800
Durable rows: 1000
{'interrupted_rows': 400, 'resumed_rows': 1000, 'rerun_rows': 1000}


## Result

The store grows from 400 rows to 1,000, then stays at 1,000 on the next run.
**Resume from the saved prefix**, the uninterrupted sequence of rows starting at zero.
This example stops cleanly, rather than cutting power, so it checks restart logic and row identity.

## Before using your own export

Use one writer and keep the vector store on a supported local filesystem.
Retain the input's identity, encoder fingerprint, and batch settings with a production job.
An encoder fingerprint identifies the model and its settings, and a changed input requires a new job.
The local store guarantee does not extend to SMB or NFS network storage.

## What next?

- [Watch a folder for new files](05-folder-listener.ipynb).
- [fse API and design](../docs/architecture/store.md)
- [Python file I/O](https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files)

Back to the [recipe index](README.md) or the [main README](../README.md).

## Clean up

The helper keeps generated files in a temporary folder. Run this after you finish exploring the results.

In [7]:
recipe.close()